# Tilapia Risk & Growth — Random Forest Models

This notebook documents how the two Random Forest models in the Smart IoT & AI Tilapia Monitoring Platform are trained and evaluated.

| Model | Task | Output |
|---|---|---|
| Random Forest Classifier (RFC) | Risk prediction | Optimal / Warning / Critical |
| Random Forest Regressor (RFR) | Growth analysis | Estimated fish weight (g) |

**Training data comes only from historical sources.** Live ESP32 readings are used for prediction, never for training.

In [ ]:
import json, sys, os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

sys.path.insert(0, os.path.abspath('.'))
import tilapia_ml as tm

## 1. Secondary historical dataset

`Data_Model_IoTMLCQ_2024.xlsx` holds hourly water-quality records for 2024. Pre-processing removes missing values and the duplicated hourly copies.

In [ ]:
hist = tm.load_excel_history()
print(f"Raw rows: {hist['raw_rows']}  ->  unique rows after cleaning: {hist['unique_rows']}")
for name, mean, std in zip(['Temperature (°C)', 'pH', 'Turbidity (NTU)'], hist['mean'], hist['std']):
    print(f'{name:18s} mean {mean:6.2f}   std {std:5.2f}')

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, i, name in zip(axes, range(3), ['Temperature (°C)', 'pH', 'Turbidity (NTU)']):
    ax.hist(hist['values'][:, i], bins=30, color='#2563eb'); ax.set_title(name)
plt.tight_layout(); plt.show()

**Limitation:** every record lies inside the safe range for tilapia, so the file contains no Warning or Critical examples, and it has no fish-growth time series. It is therefore used to describe *normal* operating conditions and as an external check, and is complemented with published tilapia knowledge below.

## 2. Knowledge-guided training datasets

* **Risk dataset:** normal samples are bootstrapped from the Excel records; stress samples cover the published tolerance ranges. Labels follow published water-quality bands plus interaction rules (toxic un-ionised ammonia, warm water with high organic load, thermal shock, unstable pH). Sensor measurement error is added to the features.
* **Growth dataset:** virtual culture batches. Daily water conditions drive the TGC growth model (Iwama & Tautz, 1981), with batch-to-batch variation, daily growth noise and ±6% weighing error at weekly sampling points.

In [ ]:
X, y, _ = tm.generate_risk_dataset(n=6000, seed=42)
print('Risk dataset:', X.shape, dict(zip(tm.CLASSES, np.bincount(y))))

Xg, yg, groups = tm.generate_growth_dataset(n_batches=500, seed=7)
print('Growth dataset:', Xg.shape, 'from', len(set(groups)), 'batches')

plt.figure(figsize=(6, 3.5))
for b in range(8):
    m = groups == b
    plt.plot(Xg[m, 1], yg[m], marker='.', lw=1)
plt.xlabel('Days in culture'); plt.ylabel('Weight (g)'); plt.title('Sample growth batches'); plt.show()

## 3. Train, tune and evaluate

`train_models.py` performs the 80/20 split (stratified for RFC, by batch for RFR), tunes hyper-parameters with 5-fold `GridSearchCV`, compares against baselines and saves the models. Run it again to retrain (about 3–6 minutes):

In [ ]:
# !python3 train_models.py
metrics = json.load(open('reports/metrics.json'))
print('Trained at', metrics['generated_at'], 'with scikit-learn', metrics['sklearn'])

## 4. Results

In [ ]:
display(Markdown(open('reports/report.md').read().split('## Figures')[0]))

In [ ]:
for f in ['rfc_confusion_matrix.png', 'model_comparison.png', 'rfr_predicted_vs_actual.png',
          'rfc_feature_importance.png', 'rfr_feature_importance.png']:
    display(Image(filename=os.path.join('reports', f)))

## 5. Using the trained models

The Laravel app calls `predict.py`, which loads `models/rf_risk_classifier.joblib` and `models/rf_growth_regressor.joblib` and feeds them each live ESP32 reading.

In [ ]:
import joblib
rfc = joblib.load('models/rf_risk_classifier.joblib')
sample = [[29.0, 4.8, 180.0, 0.0, 0.4]]   # TC-03: acidic and very murky water
proba = rfc['model'].predict_proba(sample)[0]
print(dict(zip(rfc['classes'], proba.round(3))), '->', rfc['classes'][proba.argmax()])